# #4 Simulation Plots

## Purpose

Generate plots summarizing the simulation results generated by running scripts 1-3 in the simulation folder

In [1]:
import cassiopeia as cas
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pycea as py
import seaborn as sns
import tracertools
from devmap.config import get_paths, sequential_cmap, set_theme
from devmap.plots import plot_grouped_characters
from devmap.utils import save_plot

set_theme()
base_path, plots_path, results_path = get_paths("simulation")
data_path = base_path / "data"

## Helper functions

In [3]:
def get_mutation_rate(tdata,edit_frac):
    """Calculate mutation rate from edit fraction."""
    total_time = tdata.obs.time.values[0]
    mutation_rate = 1 - (1 - edit_frac) ** (1 / total_time)
    return mutation_rate


def embryo_fitness(parent,rng = None):
    scale = 1
    if parent["time"] > 2:
        scale = .4
    return ({ "birth_scale": scale }, { "birth_rate": scale })

def hybrid(tdata):
    tdata.obst["stump"] = tracertools.solver.n_mutation_greedy(tdata.obsm["characters"])[0]
    tdata.obs["clade"] = py.get.node_df(tdata, tree = "stump")['parent']
    clade_trees = {}
    for clade in tdata.obs["clade"].dropna().unique():
        clade_characters = tdata[tdata.obs["clade"] == clade].obsm["characters"]
        use_characters = tracertools.utils.select_characters(clade_characters)
        clade_trees[clade] = tracertools.solver.fasttree(clade_characters[use_characters], root_name = clade)
    tdata.obst["hybrid"] = tracertools.tree.replace_subtrees(tdata.obst["stump"],
            list(clade_trees.values()),error_on_missing=True)

## Example tree

In [215]:
tdata = cas.sim.birth_death_process(num_extant = 5000, on_division=embryo_fitness, random_seed = 2,
    birth_waiting_distribution= lambda scale, rng: rng.lognormal(mean=np.log(scale), sigma=0.1))
cas.sim.stochastic_tracing(tdata, mutation_rate=get_mutation_rate(tdata,0.5), number_of_cassettes=35, 
    random_seed = 1,state_priors={str(i): 1 / 8 for i in range(1, 9)})
cas.tl.rescale_node_times(tdata, max = 1)
tdata.obsm["characters"].columns = (
    "intID" + tdata.obsm["characters"].columns.astype(str)
        .str.replace("-0", "-RNF2", regex=False)
        .str.replace("-1", "-HEK3", regex=False)
        .str.replace("-2", "-EMX1", regex=False)
)

Ground truth

In [216]:
fig, ax = plt.subplots(figsize=(3, 2))
py.pl.branches(tdata, ax=ax, depth_key="time", linewidth = .2)
plot_grouped_characters(tdata, ax=ax, width = 0.04)
save_plot(plots_path / "example_ground_truth.svg", fig, rasterize = True)


Reconstructed

In [57]:
hybrid(tdata)
py.pp.add_depth(tdata, tree="hybrid")
fig, ax = plt.subplots(figsize=(3, 2))
py.pl.branches(tdata, ax=ax, depth_key="depth", linewidth = .2, tree = "hybrid")
plot_grouped_characters(tdata, ax=ax, width = 0.04)
save_plot(plots_path / "example_reconstructed.svg", fig, rasterize = True)

Branch length estimates

In [ ]:
tree = tdata.obst["hybrid"].copy()
tracertools.tree.ancestral_characters(tree,tdata.obsm["characters"])
tracertools.tree.estimate_branch_lengths(tree)
tdata.obst["ble"] = tree
fig, ax = plt.subplots(figsize=(.6, 2))
py.pl.branches(tdata, ax=ax, depth_key="time", linewidth = .2, tree = "ble")
save_plot(plots_path / "example_ble.svg", fig, rasterize = True)

## Benchmarking

Load data

In [123]:
capacity = pd.read_csv(base_path / "simulation/results_capacity/benchmark_capacity_results.csv")
solver = pd.read_csv(base_path / "simulation/results_solver/benchmark_solver_results.csv")

Tracing capacity

In [ ]:

capacity["Detection\nrate (%)"] = (1 - capacity["missing_rate"]) * 100
fig, ax = plt.subplots(figsize=(2, 2))
sns.lineplot(data=capacity, x="n_characters", y="rf_norm", hue = "Detection\nrate (%)", ax=ax)
plt.xlabel("Number of edit sites")
plt.ylabel("Robinson-Foulds distance")
save_plot(plots_path / "capacity_lineplot.svg")

RF distance

In [124]:
fig, ax = plt.subplots(figsize=(2, 2))
sns.lineplot(data=solver, x="n_leaves", y="rf_norm", hue="solver", ax = ax)
plt.xscale("log");
plt.ylim(0,0.2);
plt.xlabel("Number of cells")
plt.ylabel("Robinson-Foulds distance")
save_plot(plots_path / "solver_rf_lineplot.svg")

Memory usage

In [125]:
fig, ax = plt.subplots(figsize=(2, 2))
sns.lineplot(data=solver, x="n_leaves", y="solver_mem_mb", hue="solver", ax = ax)
plt.xscale("log");
plt.yscale("log");
plt.xlabel("Number of cells")
plt.ylabel("Memory usage (MB)")
save_plot(plots_path / "solver_mem_lineplot.svg")

CPU time

In [126]:
fig, ax = plt.subplots(figsize=(2, 2))
solver["time_minutes"] = solver["time_sec"] / 60
sns.lineplot(data=solver, x="n_leaves", y="time_minutes", hue="solver", ax = ax)
plt.xscale("log");
plt.yscale("log");
plt.xlabel("Number of cells")
plt.ylabel("Processing time (minutes)")
save_plot(plots_path / "solver_time_lineplot.svg")

## Branch length estimation

In [293]:
tdata = cas.sim.birth_death_process(num_extant = 10000, on_division=embryo_fitness, random_seed = 1,
    birth_waiting_distribution= lambda scale, rng: rng.lognormal(mean=np.log(scale), sigma=0.1))
cas.sim.stochastic_tracing(tdata, mutation_rate=get_mutation_rate(tdata,0.5), number_of_cassettes=35, 
    random_seed = 1,state_priors={str(i): 1 / 8 for i in range(1, 9)})
cas.tl.rescale_node_times(tdata, max = 1)

ConvexML

In [294]:
tracertools.tree.estimate_branch_lengths(tdata.obst["simulated"], key_added = "convexml_time")
node_df = py.get.node_df(tdata).query("time != 1").copy()
fig, ax = plt.subplots(figsize=(2, 2))
sns.scatterplot(data=node_df, x="time", y="convexml_time", alpha = 0.5, s = 5, color = "gray")
mae = (abs(node_df["time"] - node_df["convexml_time"])).mean()
plt.text(0.1, 0.8, f"Mean absolute\nerror: {mae:.4f}", transform=plt.gca().transAxes)
plt.plot([0, 1], [0, 1], linestyle='--', color='black',zorder=0);
plt.xlabel("True division time")
plt.ylabel("ConvexML division time")
save_plot(plots_path / "ble_convexml_scatter.svg", rasterize = True)

laml

In [296]:
tracertools.tree.estimate_branch_lengths(tdata.obst["simulated"],method="laml", key_added = "laml_time", minimum_branch_length=0.0001)
node_df = py.get.node_df(tdata).query("time != 1").copy()
fig, ax = plt.subplots(figsize=(2, 2))
sns.scatterplot(data=node_df, x="time", y="laml_time", alpha = 0.5, s = 5, color = "gray")
mae = (abs(node_df["time"] - node_df["laml_time"])).mean()
plt.text(0.1, 0.8, f"Mean absolute\nerror: {mae:.4f}", transform=plt.gca().transAxes)
plt.plot([0, 1], [0, 1], linestyle='--', color='black',zorder=0);
plt.xlabel("True division time")
plt.ylabel("LAML-Pro division time")
save_plot(plots_path / "ble_laml_scatter.svg", rasterize = True)

In [287]:
comparison = []
for i in range(10):
    print(f"Running simulation {i}")
    tdata = cas.sim.birth_death_process(num_extant = 10000, on_division=embryo_fitness, random_seed = i,
        birth_waiting_distribution= lambda scale, rng: rng.lognormal(mean=np.log(scale), sigma=0.1))
    cas.sim.stochastic_tracing(tdata, mutation_rate=get_mutation_rate(tdata,0.5), number_of_cassettes=35, 
        random_seed = i,state_priors={str(i): 1 / 8 for i in range(1, 9)})
    cas.tl.rescale_node_times(tdata, max = 1)
    tracertools.tree.estimate_branch_lengths(tdata.obst["simulated"], key_added = "convexml_time")
    tracertools.tree.estimate_branch_lengths(tdata.obst["simulated"],method="laml", key_added = "laml_time", minimum_branch_length=0.001)
    node_df = py.get.node_df(tdata).query("time != 1").copy()
    convexml_mae = (abs(node_df["time"] - node_df["convexml_time"])).mean()
    laml_mae = (abs(node_df["time"] - node_df["laml_time"])).mean()
    comparison.append((convexml_mae, laml_mae, i))
comparison = pd.DataFrame(comparison, columns=["ConvexML MAE", "LAML MAE", "Seed"])

In [288]:
comparison_long = comparison.melt(id_vars=["Seed"], value_vars=["ConvexML MAE", "LAML MAE"], 
        var_name="Method", value_name="MAE")
comparison_long["Method"] = comparison_long["Method"].str.replace(" MAE", "")
fig, ax = plt.subplots(figsize=(1.5, 2))
sns.barplot(data=comparison_long, x="Method", y="MAE", color = "lightgray",
    err_kws={"linewidth": 1,"color":"black"},capsize=.1,
    edgecolor="black",linewidth=0.8,)
plt.ylabel("Mean absolute error")
plt.xlabel("")
save_plot(plots_path / "mae_barplot.svg")

benchmarking

In [223]:
ble = pd.read_csv(base_path / "simulation/results_ble/benchmark_ble_results.csv")
fig, ax = plt.subplots(figsize=(2, 2))
sns.lineplot(data = ble, x = "n_characters", y = "mae", hue="division_sigma", palette="crest")
plt.xlabel("Number of edit sites")
plt.ylabel("Mean absolute error")
save_plot(plots_path / "ble_lineplot.svg")